In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Pearson Correlation Heatmap for Raw Triage Features (`plots/pearson_correlation.ipynb`)

This notebook computes the pairwise **Pearson Correlation Matrix** across **ONLY the raw features** defined in `config/triage_conf.json` without any feature engineering:

### Raw Input Features (From `triage_conf.json`)
- `age`, `gender`, `cc_breathingdifficulty`
- `triage_vital_hr`, `triage_vital_sbp`, `triage_vital_rr`, `triage_vital_o2`
- `pulse_last`, `resp_last`, `spo2_last`, `sbp_last`
- `pulse_min`, `resp_min`, `spo2_min`, `sbp_min`
- `pulse_max`, `resp_max`, `spo2_max`, `sbp_max`

### Workflow & Outputs
1. **Config & Raw Feature Extraction**: Parses `triage_conf.json` and loads raw features directly from dataset without feature engineering.
2. **Pearson Correlation Calculation**: Computes exact pairwise linear correlation coefficients $r = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum (x_i - \bar{x})^2 \sum (y_i - \bar{y})^2}}$ across all raw feature pairs.
3. **Heatmap Visualization**: Generates a high-resolution, color-coded correlation heatmap with numeric overlays saved to `plots/pearson_correlation_heatmap.png`.
4. **CSV Export**: Writes the full symmetric $19 \times 19$ matrix report to `reports/pearson_correlation_matrix.csv`.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(dplyr)
  library(ggplot2)
  library(tidyr)
  library(reshape2)
})
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}
config <- fromJSON(config_path)
cat("=== Configuration Loaded from config/triage_conf.json ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Raw Features Listed in JSON:\n")
print(config$features$data_name)

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Dataset & Extract Raw Features ONLY (No Feature Engineering)
# ---------------------------------------------------------
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}
cat("Loading dataset from:", data_file, "...\n")
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
cat(sprintf("Selected main dataset object: '%s' (%d rows)\n", data_obj_name, max(df_sizes)))
raw_df <- get(data_obj_name, envir = data_env)
raw_feature_cols <- config$features$data_name
# Extract raw features directly without feature engineering
df_raw_features <- data.frame(matrix(ncol = 0, nrow = nrow(raw_df)))
for (col_name in raw_feature_cols) {
  if (col_name == "gender") {
    df_raw_features$gender <- ifelse(as.character(raw_df$gender) == "Male", 1, 0)
  } else if (col_name == "cc_breathingdifficulty") {
    df_raw_features$cc_breathingdifficulty <- ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0)
  } else {
    vec <- raw_df[[col_name]]
    vec[is.na(vec)] <- 0
    df_raw_features[[col_name]] <- vec
  }
}
df_features <- na.omit(df_raw_features)
cat(sprintf("Complete cases ready for correlation analysis: %d rows x %d raw features\n", nrow(df_features), ncol(df_features)))
cat("Raw Features Included (19 Total):\n")
print(names(df_features))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Compute Pearson Correlation Matrix & Export CSV Report
# ---------------------------------------------------------
cor_matrix <- cor(df_features, method = "pearson", use = "pairwise.complete.obs")
reports_dir <- "../reports"
if (!dir.exists(reports_dir)) reports_dir <- "reports"
if (!dir.exists(reports_dir)) dir.create(reports_dir, recursive = TRUE)
csv_path <- file.path(reports_dir, "pearson_correlation_matrix.csv")
write.csv(cor_matrix, file = csv_path, row.names = TRUE)
cat("Full Pearson Correlation Matrix (Raw Features) written to CSV:", csv_path, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Render & Save Heatmap Plot
# ---------------------------------------------------------
cor_melted <- melt(cor_matrix)
colnames(cor_melted) <- c("Feature1", "Feature2", "Correlation")
p_heat <- ggplot(cor_melted, aes(x = Feature1, y = Feature2, fill = Correlation)) +
  geom_tile(color = "white", linewidth = 0.3) +
  scale_fill_gradient2(low = "#2b5c8f", mid = "#ffffff", high = "#e07a5f", midpoint = 0, limit = c(-1, 1), name = "Pearson\nCorrelation") +
  geom_text(aes(label = sprintf("%.2f", Correlation)), color = "black", size = 2.5) +
  theme_minimal() +
  labs(title = "Pearson Correlation Matrix Heatmap (Raw Features ONLY)",
       subtitle = "Pairwise Linear Feature Correlation Analysis for triage_conf.json Raw Inputs",
       x = "", y = "") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1, size = 9, face = "bold"),
    axis.text.y = element_text(size = 9, face = "bold"),
    plot.title = element_text(face = "bold", size = 14, hjust = 0.5),
    plot.subtitle = element_text(size = 11, hjust = 0.5),
    legend.position = "right"
  )
plots_dir <- "../plots"
if (!dir.exists(plots_dir)) plots_dir <- "plots"
if (!dir.exists(plots_dir)) dir.create(plots_dir, recursive = TRUE)
plot_file <- file.path(plots_dir, "pearson_correlation_heatmap.png")
ggsave(plot_file, plot = p_heat, width = 12, height = 10, dpi = 300)
cat("Pearson Correlation Heatmap saved to:", plot_file, "\n")
p_heat